## Imports

In [ ]:
# import the libraries used in this project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load the Dataset

In [ ]:
# load the dataset from excel
df = pd.read_excel(r"C:\Users\NelsonP\Documents\IronHack\FinalProject\data\dataset_final_project.xlsx", header=1)
df.head()

In [ ]:
# check the number of rows and columns
df.shape

In [ ]:
# check column types and missing values
df.info()

## Data Preparation

This section covers all the steps needed to turn the raw dataset into a clean,
reliable base for analysis and for the replenishment logic. This includes
selecting and renaming columns, filtering out warehouses and product families
that are out of scope, and fixing formatting or consistency issues in the data
(missing values, negative quantities, inconsistent brand names, etc.).

### Column Selection and Renaming

In [ ]:
# select only the columns needed for this project
useful_columns = ['MARCA', 'ARM', 'REF', 'DESCRICAO', 'TAMANHO', 'LOCALIZACAO', 'QUANTIDADE', 'TIPO ARTIGO','FAMILIA']
df_work = df[useful_columns].copy()
df_work.head()

In [ ]:
# rename columns to english
df_work = df_work.rename(columns={
    'MARCA': 'BRAND',
    'ARM': 'WAREHOUSE',
    'REF': 'REFERENCE',
    'DESCRICAO': 'DESCRIPTION',
    'TAMANHO': 'SIZE',
    'LOCALIZACAO': 'LOCATION',
    'QUANTIDADE': 'QUANTITY',
    'TIPO ARTIGO': 'ITEM_TYPE',
    'FAMILIA': 'FAMILY'
})

df_work.head()

### Warehouse Filtering

In [ ]:
# check how many rows exist per warehouse
df_work['WAREHOUSE'].value_counts().sort_index()

The dataset contains multiple warehouse codes. For this project, only three are relevant:

- **Warehouse 1**: central warehouse (sede) — the stock source for replenishment
- **Warehouse 2**: stores — the core of the business
- **Warehouse 5**: outlets — stores where old or discounted stock is sold

All other warehouse codes are excluded, as they are out of scope for the replenishment logic.

In [ ]:
# keep only the relevant warehouses
df_work = df_work[df_work['WAREHOUSE'].isin([1, 2, 5])].copy()

df_work['WAREHOUSE'].value_counts().sort_index()

### Location Cleaning and Store ID

In [ ]:
# check how many unique locations exist
df_work['LOCATION'].nunique()

In [ ]:
# check how many unique locations exist per warehouse
df_work.groupby('WAREHOUSE')['LOCATION'].nunique()

`LOCATION` is missing for every row in Warehouse 1 (the central warehouse). This is
not a data error — the central warehouse does not use the same shelf/location system
as a store, so it has no location value. These missing values are unified with the
existing `'HQ'` label, and used to build a `STORE_ID` that identifies each
warehouse/location combination.

In [ ]:
# convert numeric locations to integers, and keep 'HQ' as is
df_work['LOCATION'] = df_work['LOCATION'].apply(
    lambda x: str(int(x)) if pd.notna(x) and x != 'HQ' else 'HQ'
)

# build a store id from warehouse and location
df_work['STORE_ID'] = df_work['WAREHOUSE'].astype(str) + '-' + df_work['LOCATION'].astype(str)
df_work['STORE_ID'].value_counts()

### Quantity Cleaning

In [ ]:
# preview raw quantity values
df_work['QUANTITY'].unique()[:20]

In [ ]:
# convert quantity to float (portuguese decimal comma to dot)
df_work['QUANTITY'] = df_work['QUANTITY'].astype(str).str.replace(',', '.').astype(float)

df_work['QUANTITY'].dtype

In [ ]:
# count negative quantity values
df_work[(df_work['WAREHOUSE'].isin([1, 2, 5])) & (df_work['QUANTITY'] < 0)].shape[0]

**Handling negative quantities**

Negative values were found in `QUANTITY`, including 993 records (2.7%) at the
central warehouse (Warehouse 1), likely caused by incomplete transfers or
unregularized stock movements. Since the replenishment logic depends only on
central warehouse availability (not store-level quantity), negative values are
treated as 0. This is a conservative simplification: in the worst case, the
system withholds a replenishment that might have been possible, never the
opposite.

In [ ]:
# replace negative quantity values with 0
df_work['QUANTITY'] = df_work['QUANTITY'].apply(lambda x: max(x, 0))

In [ ]:
# confirm no negative values remain
(df_work['QUANTITY'] < 0).sum()

### Family Filtering — Footwear Only

In [ ]:
# check the unique values in the family column
df_work['FAMILY'].value_counts()

In [ ]:
# check how many records have no family assigned
df_work['FAMILY'].isna().sum()

The dataset includes multiple product families (footwear, apparel, textiles,
accessories) with 194 distinct `FAMILY` values. This project's scope is limited
to footwear. Records are kept only when `FAMILY` explicitly contains
`"CALÇADO"` (covering men's, women's, unisex, and children's variants).
Records with a missing `FAMILY` (110 rows) are excluded as well, since footwear
cannot be confirmed for them without ambiguity — a conservative choice that
keeps the dataset consistent and easy to justify.

In [ ]:
# keep only records explicitly marked as footwear (drops blanks and other categories)
df_work = df_work[df_work['FAMILY'].str.contains('CALÇADO', case=False, na=False)].copy()

df_work['FAMILY'].value_counts()

In [ ]:
# unify all footwear variants into a single label
df_work['FAMILY'] = 'CALÇADO'

df_work['FAMILY'].value_counts()

### Reference and Brand Consistency

In [ ]:
# check reference and brand columns for formatting issues (whitespace, case)
ref_whitespace = df_work['REFERENCE'].astype(str).apply(lambda x: x != x.strip()).sum()
ref_dup_ci = df_work['REFERENCE'].astype(str).str.upper().duplicated().sum()
ref_dup_exact = df_work['REFERENCE'].astype(str).duplicated().sum()
print(f"REFERENCE — whitespace: {ref_whitespace}, case-insensitive dupes: {ref_dup_ci}, exact dupes: {ref_dup_exact}")

brand_whitespace = df_work['BRAND'].astype(str).apply(lambda x: x != x.strip()).sum()
brand_unique_exact = df_work['BRAND'].nunique()
brand_unique_ci = df_work['BRAND'].astype(str).str.upper().nunique()
print(f"BRAND — whitespace: {brand_whitespace}, unique (exact): {brand_unique_exact}, unique (case-insensitive): {brand_unique_ci}")

**Note:** Reference codes were checked for whitespace and case inconsistencies
(none found). Character-transposition typos (e.g. "AB123" vs "BA123") were not
exhaustively checked, as this is out of scope given the project timeline.

Reference-level matching is a planned extension for a later phase, once the
chatbot can support more specific queries (e.g. "replenish reference ABC123").
For now, replenishment logic works at the brand level.

### Brand Name Fixes

In [ ]:
# inspect the fila apparel_acc brand entries
df_work[df_work['BRAND'] == 'Fila apparel_acc'][['REFERENCE', 'DESCRIPTION', 'SIZE', 'FAMILY']]

The `DESCRIPTION` and `SIZE` values confirm these are footwear items, not
apparel or accessories — the brand name just has a stray suffix from the
source system. This is fixed by merging it into the main `Fila` brand.

In [ ]:
# fix inconsistent brand naming for fila
df_work['BRAND'] = df_work['BRAND'].replace('Fila apparel_acc', 'Fila')

# confirm the fix
df_work[df_work['BRAND'].str.contains('Fila', case=False, na=False)]['BRAND'].unique()

In [ ]:
# inspect munich and merrell brand entries
df_work[df_work['BRAND'].isin(['Munich', 'Munich Sports'])][['REFERENCE', 'DESCRIPTION']].head(10)
df_work[df_work['BRAND'].isin(['Merrell Foot', 'Merrell Aces'])][['REFERENCE', 'DESCRIPTION']].head(10)

In [ ]:
# fix inconsistent brand naming for munich and merrell
df_work['BRAND'] = df_work['BRAND'].replace({
    'Merrell Aces': 'Merrell Foot',
    'Munich Sports': 'Munich'
})

# confirm the fix
df_work[df_work['BRAND'].str.contains('Merrell', case=False, na=False)]['BRAND'].unique()
df_work[df_work['BRAND'].str.contains('Munich', case=False, na=False)]['BRAND'].unique()

## Data Visualization

This section uses charts to make data-driven decisions about which stores and
brands to work with, instead of picking them arbitrarily.

### Store Selection

Stores/outlets with less than 100 total units in stock were excluded from the
analysis — likely temporary locations, pop-up events, or stores with
residual/discontinued stock, not representative of regular replenishment
activity. From the remaining stores, the top 6 by total quantity were
selected as the working scope for this project. For chart readability, these
6 stores are given fictional but realistic display names, unrelated to real
locations (the underlying data stays anonymized).

In [ ]:
# total footwear stock per store (excludes hq, which dwarfs individual stores)
store_quantity = df_work[df_work['WAREHOUSE'].isin([2, 5])].groupby('STORE_ID')['QUANTITY'].sum().sort_values(ascending=False)

# drop stores with negligible stock (below 100 units total)
store_quantity_filtered = store_quantity[store_quantity >= 100]

print(f"Stores removed (below 100 units): {(store_quantity < 100).sum()}")
print(f"Stores remaining: {len(store_quantity_filtered)}")

In [ ]:
# create display names for the selected stores
store_names = {
    '5-52': 'Outlet Cascais',
    '5-53': 'Outlet Gaia',
    '5-50': 'Outlet Loures',
    '2-8': 'Loja Amoreiras',
    '5-51': 'Outlet Almada',
    '2-41': 'Loja Colombo',
}

# apply the mapping only for display, keeping STORE_ID as the real key
store_quantity_filtered.index = store_quantity_filtered.index.map(lambda x: store_names.get(x, x))

In [ ]:
# plot total footwear stock for the top 6 selected stores
top_n = 6
colors = ['#2E86AB' if i < top_n else '#D3D3D3' for i in range(len(store_quantity_filtered))]

plt.figure(figsize=(10, 8))
bars = plt.barh(store_quantity_filtered.index, store_quantity_filtered.values, color=colors)
plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 50, bar.get_y() + bar.get_height()/2, f'{int(width):,}', 
              va='center', fontsize=9)

plt.xlabel('Total Quantity', fontsize=11)
plt.title('Total Footwear Stock by Store — Top 6 Selected', fontsize=13, fontweight='bold')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# filter the dataset to the selected stores, keeping hq as the stock source
SELECTED_STORES = ['5-52', '5-53', '5-50', '2-8', '5-51', '2-41']

df_work = df_work[df_work['STORE_ID'].isin(SELECTED_STORES + ['1-HQ'])].copy()

df_work['STORE_ID'].value_counts()

### Brand Representation

**Scope decision:** replenishment logic works at the brand level, not
individual reference/SKU, for this phase of the project. This matches a
simpler first version of the chatbot interaction ("replenish brand X at
store Y") and avoids reference-matching issues while the core flow is being
built.

In [ ]:
# plot brand representation across the selected stores (excluding hq)
brand_quantity = df_work[df_work['STORE_ID'] != '1-HQ'].groupby('BRAND')['QUANTITY'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
bars = plt.barh(brand_quantity.index, brand_quantity.values, color='#2E86AB')
plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 20, bar.get_y() + bar.get_height()/2, f'{int(width):,}', 
              va='center', fontsize=9)

plt.xlabel('Total Quantity', fontsize=11)
plt.title('Footwear Stock by Brand — Selected Stores', fontsize=13, fontweight='bold')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

### Brand Distribution Across Stores

In [ ]:
# build brand by store matrix, using display names for readability
pivot = df_work[df_work['STORE_ID'] != '1-HQ'].pivot_table(
    index='BRAND', columns='STORE_ID', values='QUANTITY', aggfunc='sum', fill_value=0
)

# apply the display names to columns (store_id stays the real key elsewhere)
pivot.columns = pivot.columns.map(lambda x: store_names.get(x, x))

plt.figure(figsize=(8, 10))
sns.heatmap(pivot, cmap='Blues', annot=True, fmt='.0f', cbar_kws={'label': 'Quantity'})
plt.title('Brand Distribution Across Selected Stores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Replenishment Availability by Brand

This section checks, for each brand, whether the central warehouse (HQ) has
any stock available — since that is the only thing the replenishment logic
depends on.

In [ ]:
# rebuild the matrix keeping hq, using real store_id (not display names)
pivot = df_work.pivot_table(
    index='BRAND', columns='STORE_ID', values='QUANTITY', aggfunc='sum', fill_value=0
)

# check, for each brand, whether hq has any stock available
hq_stock = pivot['1-HQ']
can_replenish = hq_stock > 0

replenish_status = pd.DataFrame({
    'HQ_Stock': hq_stock,
    'Can_Replenish': can_replenish.map({True: 'Yes', False: 'No — HQ out of stock'})
}).sort_values('HQ_Stock', ascending=False)

replenish_status

In [ ]:
# visual version: green = can replenish, red = hq out of stock
plt.figure(figsize=(10, 10))
sorted_stock = hq_stock.sort_values(ascending=False)
colors = ['#2ECC71' if x > 0 else '#E74C3C' for x in sorted_stock]

bars = plt.barh(sorted_stock.index, sorted_stock.values, color=colors)
plt.gca().invert_yaxis()

for bar, value in zip(bars, sorted_stock.values):
    label = f'{int(value):,}' if value > 0 else 'Out of stock'
    x_pos = bar.get_width() + (sorted_stock.max() * 0.01) if value > 0 else 5
    plt.text(x_pos, bar.get_y() + bar.get_height()/2, label, va='center', fontsize=8)

plt.xlabel('HQ Stock Available', fontsize=11)
plt.title('Replenishment Availability by Brand (HQ Stock)', fontsize=13, fontweight='bold')
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
# summary: how many brands can vs cannot be replenished from hq
summary_counts = can_replenish.value_counts()
summary_counts.index = summary_counts.index.map({True: 'Can Replenish', False: 'HQ Out of Stock'})

plt.figure(figsize=(6, 6))
colors = ['#2ECC71', '#E74C3C']
plt.pie(summary_counts.values, labels=summary_counts.index, autopct='%1.0f%%', 
        colors=colors, startangle=90, textprops={'fontsize': 12})
plt.title('Brands: Replenishable from HQ?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()